# Assignment 3: Numerical Data Handling

**Professor:** Subodh Kumar  
**Assignment:** 3  
**Marks:** 100

## Objective

Download a relevant dataset from **UCI Machine Learning Repository or Kaggle** and perform numerical data handling.

This notebook follows the numerical-data preprocessing topics from **Chapter 4 of O'Reilly's Machine Learning with Python Cookbook**.

## Dataset Selected

### Predict Students' Dropout and Academic Success

The dataset contains academic, demographic and macroeconomic information about students. For this assignment, only the variables that are genuinely numerical are selected for numerical data handling.

The target variable `Target` is kept separately and is not used as a numerical feature.

### Numerical Data Handling Covered

1. Dataset loading and inspection
2. Identification of genuine numerical features
3. Missing-value analysis
4. Descriptive statistics
5. Min-Max scaling
6. Standardization
7. L1 normalization
8. Robust scaling
9. Log transformation
10. Polynomial and interaction features
11. IQR-based outlier detection
12. Outlier capping
13. Histograms
14. Box plots
15. Correlation heatmap
16. Before/after comparison


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    Normalizer,
    RobustScaler,
    PolynomialFeatures
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Load the Dataset

The dataset is loaded from a local CSV file.

If the CSV is stored in the same folder as this notebook, keep the filename below. Otherwise, change `file_path` to the correct location.

In [ ]:
file_path = "dataset.csv"

df = pd.read_csv(
    file_path,
    sep=None,
    engine="python"
)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

## 3. Initial Dataset Inspection

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", df.duplicated().sum())

## 4. Identify Genuine Numerical Features

A column can be stored as an integer but still represent a categorical code. Therefore, numerical data handling should not blindly apply to every integer column.

The following variables are treated as genuine numerical measurements in this dataset.

In [ ]:
numerical_features = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

missing_columns = [
    column for column in numerical_features
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"These expected numerical columns are missing: {missing_columns}"
    )

numerical_data = df[numerical_features].copy()

print("Number of genuine numerical features:", len(numerical_features))
display(numerical_data.head())

## 5. Descriptive Statistics

Descriptive statistics help us understand the scale and distribution of numerical variables before applying preprocessing techniques.

In [ ]:
summary = numerical_data.describe().T

summary["range"] = summary["max"] - summary["min"]

display(summary.round(3))

## 6. Check Missing Values in Numerical Features

In [ ]:
missing_numeric = numerical_data.isnull().sum()

print("Missing numerical values:")
display(
    missing_numeric[missing_numeric > 0]
    .to_frame("Missing Values")
)

print("Total missing numerical values:",
      missing_numeric.sum())

## 7. Min-Max Scaling

Min-Max scaling converts a numerical feature to a fixed range, normally 0 to 1.

$$
x' = \frac{x-x_{min}}{x_{max}-x_{min}}
$$

This is useful when numerical features have very different ranges.

In [ ]:
minmax_scaler = MinMaxScaler()

minmax_data = pd.DataFrame(
    minmax_scaler.fit_transform(numerical_data),
    columns=numerical_features,
    index=numerical_data.index
)

display(minmax_data.head())

print("Minimum value after Min-Max scaling:")
display(minmax_data.min().round(3).head())

print("\nMaximum value after Min-Max scaling:")
display(minmax_data.max().round(3).head())

## 8. Standardization

Standardization centers each feature around zero and scales it using its standard deviation.

$$
z = \frac{x-\mu}{\sigma}
$$

In [ ]:
standard_scaler = StandardScaler()

standardized_data = pd.DataFrame(
    standard_scaler.fit_transform(numerical_data),
    columns=numerical_features,
    index=numerical_data.index
)

display(standardized_data.head())

print("Means after standardization:")
display(standardized_data.mean().round(3).head())

print("\nStandard deviations after standardization:")
display(standardized_data.std(ddof=0).round(3).head())

## 9. L1 Normalization

Normalization operates on each observation rather than independently on each feature.

For L1 normalization, the sum of the absolute values in each row is 1.

In [ ]:
normalizer = Normalizer(norm="l1")

l1_data = pd.DataFrame(
    normalizer.fit_transform(numerical_data),
    columns=numerical_features,
    index=numerical_data.index
)

display(l1_data.head())

print("Row-wise L1 sums:")
display(l1_data.abs().sum(axis=1).head())

## 10. Robust Scaling

RobustScaler uses the median and interquartile range. It is less sensitive to extreme observations than methods based on the mean and standard deviation.

In [ ]:
robust_scaler = RobustScaler()

robust_data = pd.DataFrame(
    robust_scaler.fit_transform(numerical_data),
    columns=numerical_features,
    index=numerical_data.index
)

display(robust_data.head())

## 11. Log Transformation

A logarithmic transformation can reduce right-skewness for positive variables.

Here, `Admission grade` is transformed using:

$$
x' = \log(1+x)
$$

The original feature is preserved.

In [ ]:
if (df["Admission grade"] < 0).any():
    raise ValueError("Admission grade contains negative values.")

df["Log Admission grade"] = np.log1p(
    df["Admission grade"]
)

display(
    df[
        ["Admission grade", "Log Admission grade"]
    ].head()
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    df["Admission grade"],
    bins=30
)
axes[0].set_title("Original Admission Grade")
axes[0].set_xlabel("Admission Grade")
axes[0].set_ylabel("Frequency")

axes[1].hist(
    df["Log Admission grade"],
    bins=30
)
axes[1].set_title("Log-Transformed Admission Grade")
axes[1].set_xlabel("Log Admission Grade")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 12. Polynomial and Interaction Features

Polynomial features allow numerical models to represent nonlinear relationships.

Two meaningful numerical variables are used here:

- Admission grade
- Previous qualification grade

A degree-2 transformation creates squared terms and their interaction.

In [ ]:
poly_input = df[
    [
        "Admission grade",
        "Previous qualification (grade)"
    ]
].copy()

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

poly_array = poly.fit_transform(poly_input)

poly_data = pd.DataFrame(
    poly_array,
    columns=poly.get_feature_names_out(poly_input.columns)
)

display(poly_data.head())

print("Polynomial feature shape:", poly_data.shape)

## 13. Outlier Detection Using the IQR Method

The Interquartile Range is:

$$
IQR = Q3-Q1
$$

An observation is treated as an outlier if it lies below:

$$
Q1-1.5(IQR)
$$

or above:

$$
Q3+1.5(IQR)
$$

In [ ]:
outlier_results = []

for column in numerical_features:
    q1 = numerical_data[column].quantile(0.25)
    q3 = numerical_data[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = (
        (numerical_data[column] < lower_bound) |
        (numerical_data[column] > upper_bound)
    )

    outlier_results.append({
        "Feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outliers.sum()),
        "Outlier %": outliers.mean() * 100
    })

outlier_summary = pd.DataFrame(outlier_results)

display(outlier_summary.round(3))

## 14. Visualize Outliers with Box Plots

Box plots provide a visual representation of the median, quartiles and potential extreme observations.

In [ ]:
plot_columns = [
    "Admission grade",
    "Previous qualification (grade)",
    "Age at enrollment",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (grade)"
]

plt.figure(figsize=(13, 6))

sns.boxplot(
    data=numerical_data[plot_columns]
)

plt.title("Box Plots of Selected Numerical Features")
plt.xticks(rotation=20)
plt.ylabel("Value")
plt.tight_layout()
plt.show()

## 15. Outlier Capping

Instead of deleting observations, extreme values are capped at the IQR boundaries.

This keeps the observations while reducing the influence of extreme values.

In [ ]:
capped_data = numerical_data.copy()

for column in numerical_features:
    q1 = capped_data[column].quantile(0.25)
    q3 = capped_data[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    capped_data[column] = capped_data[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Original shape:", numerical_data.shape)
print("Capped data shape:", capped_data.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(
    y=numerical_data["Admission grade"],
    ax=axes[0]
)
axes[0].set_title("Admission Grade Before Capping")

sns.boxplot(
    y=capped_data["Admission grade"],
    ax=axes[1]
)
axes[1].set_title("Admission Grade After Capping")

plt.tight_layout()
plt.show()

## 16. Correlation Analysis

Correlation analysis helps identify linear relationships among numerical variables.

The correlation coefficient ranges from -1 to +1.

In [ ]:
correlation_matrix = numerical_data.corr()

plt.figure(figsize=(15, 11))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    annot=False
)

plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

## 17. Compare Scaling Methods

The same feature is displayed in its original, Min-Max scaled, standardized and robust-scaled forms.

In [ ]:
comparison = pd.DataFrame({
    "Original": numerical_data["Admission grade"].head(10).values,
    "Min-Max": minmax_data["Admission grade"].head(10).values,
    "Standardized": standardized_data["Admission grade"].head(10).values,
    "Robust": robust_data["Admission grade"].head(10).values
})

display(comparison.round(4))

## 18. Numerical Data Before and After Scaling

A visual comparison helps demonstrate how standardization changes the numerical distribution while preserving the relative pattern.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    numerical_data["Admission grade"],
    bins=30
)
axes[0].set_title("Original Admission Grade")

axes[1].hist(
    standardized_data["Admission grade"],
    bins=30
)
axes[1].set_title("Standardized Admission Grade")

plt.tight_layout()
plt.show()

## 19. Final Numerical Dataset

The capped numerical data can be used as a cleaned numerical feature matrix for subsequent analysis or machine learning.

The target variable remains separate.

In [ ]:
X_numerical = capped_data.copy()

if "Target" in df.columns:
    y = df["Target"].copy()
else:
    y = None

print("Final numerical feature matrix shape:", X_numerical.shape)

display(X_numerical.head())

if y is not None:
    print("\nTarget distribution:")
    display(y.value_counts())

# 20. Summary of Numerical Data Handling

| Technique | Completed |
|---|---|
| Dataset loading | ✓ |
| Numerical feature identification | ✓ |
| Missing-value analysis | ✓ |
| Descriptive statistics | ✓ |
| Min-Max scaling | ✓ |
| Standardization | ✓ |
| L1 normalization | ✓ |
| Robust scaling | ✓ |
| Log transformation | ✓ |
| Polynomial features | ✓ |
| Interaction features | ✓ |
| IQR outlier detection | ✓ |
| Outlier capping | ✓ |
| Histogram | ✓ |
| Box plot | ✓ |
| Correlation heatmap | ✓ |
| Before/after comparison | ✓ |

## Conclusion

Numerical data can have different scales, distributions and extreme observations. Therefore, numerical preprocessing is an important step before statistical analysis and machine learning.

In this assignment, Min-Max scaling, standardization, normalization and robust scaling were applied to understand different approaches to numerical transformation. Log transformation was used to transform a positive feature, while polynomial features were used to create nonlinear and interaction terms. Outliers were detected using the IQR method and handled through capping. Histograms, box plots and a correlation heatmap were used to visually examine the numerical variables.

These operations demonstrate how numerical data can be prepared systematically for further analysis.

# References

1. **UCI Machine Learning Repository – Predict Students' Dropout and Academic Success**  
   https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

2. Cortez, P. & Silva, A. M. G. (2012). *Using Data Mining to Predict Secondary School Student Performance*.

3. *Machine Learning with Python Cookbook*, O'Reilly Media — Chapter 4: Handling Numerical Data.
